## Complete and Intuitive Guide to the Particle Filter

### A Step-by-Step "From Scratch" Explanation for Non-Gaussian Localization of a Mobile Robot (Duckiebot)

Welcome to the second notebook in the probabilistic localization series! In the previous notebook, we studied the **Kalman Filter (KF)**. The Kalman Filter is a very powerful mathematical tool, but it has one major "Achilles' heel":

* It works **only** if the world is **linear** and if uncertainty can be represented by **a single Gaussian bell curve (unimodal)**.

### The Limitation of the Kalman Filter in the Real World

Imagine your Duckiebot enters a symmetrical rectangular room with two identical doors. The sensor reads: *"I see a door 1 meter away."*

* Where is the robot? It could be in front of **Door A**, or in front of **Door B**!
* The true probability distribution has **two peaks** (a bimodal/multimodal distribution).
* The Kalman Filter cannot handle two peaks: it is forced to compute the average between the two doors, placing its estimate... **right in the middle of the wall, where there is no door at all!**

### The Revolutionary Solution: The Particle Filter (Monte Carlo Method)

Instead of describing the position with a rigid mathematical formula (mean and covariance), the **Particle Filter (PF)** uses a brilliant and intuitive strategy:

> **We release an "army" of $M$ virtual robots (called particles or hypotheses) into the environment.**

Each particle $i$ holds:

1. A **hypothesized state** $\mathbf{x}_t^i$ (e.g., an $(x, y)$ coordinate on the floor).
2. An **importance weight** $w_t^i$ (a score that answers: *"How credible is this particle given the real sensor measurements?"*).

In this guide, we will break down the Particle Filter step by step using concrete Duckiebot examples, formulas explained in simple terms, and working Python code commented line by line.

### Code Explanation: Importing Libraries & Setting Seeds

* **Purpose:** Sets up the Python environment for numerical computation, statistical modeling, and plotting.
* **Key Details:**
* `numpy` and `scipy.stats` handle matrix math and probability density functions (PDFs).
* `matplotlib` is used to visualize the robot's trajectory and sensor noise.
* `np.random.seed(42)` and `random.seed(42)` fix the random number generators so that the simulation produces reproducible results every time it runs—ideal for classroom demonstrations.



---

## The 4 Fundamental Steps of the Particle Filter Algorithm

The lifecycle of a Particle Filter repeats at every time step $t$ through 4 sequential phases:

```
  [1. Initialization] ---> [2. Prediction / Motion]
                                    |
                                    v
  [4. Resampling]     <--- [3. Measurement / Weights]

```

1. **Initialization (Initial Sampling):**
* At the beginning, we do not know where the Duckiebot is on the track.
* We generate $M$ particles distributed **uniformly and randomly** across the entire map.
* Each particle is initially assigned the same weight: $w_0^i = 1$ (or $w_0^i = 1/M$).


2. **Prediction (Prediction Step / Motion Update):**
* We send a control command $\mathbf{u}_t$ to the Duckiebot (e.g., velocity $v_x, v_y$ for 1 second).
* We move **each particle** forward according to the motion physics, adding a bit of **random noise** to each one to simulate wheel slippage:

$$\mathbf{x}_t^i = \mathbf{x}_{t-1}^i + \mathbf{u}_t + \mathbf{w}_t^i, \quad \mathbf{w}_t^i \sim \mathcal{N}(0, \sigma_R^2 I)$$


* The particles move, and the "cloud" spreads out slightly.


3. **Weight Update (Measurement Update):**
* The real robot acquires a measurement $\mathbf{z}_t$ using its sensors.
* We now query every single particle: *"If you were the robot, how likely would it be to read this exact measurement $\mathbf{z}_t$?"*
* This probability is the **likelihood** $p(\mathbf{z}_t \vert{} \mathbf{x}_t^i)$.
* The particle's new weight becomes proportional to this value:

$$w_t^i = p(\mathbf{z}_t \vert{} \mathbf{x}_t^i)$$


* Particles close to the robot's true position get very high weights, while distant particles receive weights close to zero.


4. **Resampling (Roulette Wheel Selection):**
* We eliminate the particles that "guessed poorly" (low weight) and duplicate those that "guessed well" (high weight).
* We draw $M$ new particles from the old set with a probability proportional to their weights $w_t^i$.
* The particle cloud rapidly collapses around the true position of the robot!


5. **Final State Estimation:**
* The estimated position of the robot is obtained by computing the **weighted average** (or the arithmetic mean of the resampled particles):

$$\hat{\mathbf{x}}_t = \frac{1}{M} \sum_{i=1}^M \mathbf{x}_t^i$$





---

## The Challenge of *Particle Depletion* and the $\alpha$ Solution

Pure resampling introduces a potential issue:

* If no particles were initially generated near the robot's true position, or if the robot is suddenly picked up and moved by a human (the *Kidnapped Robot* problem), all existing particles will have very low weights.
* After a few resampling steps, all particles collapse onto a single wrong location and **die out without being able to recover** (*Particle Depletion*).

### The Random Injection Trick (Parameter $\alpha$)

To solve this problem effectively:

* We resample a majority percentage (e.g., $\alpha = 95\% = 0.95$) of particles based on the current particle weights.
* We generate the remaining $(1 - \alpha) = 5\%$ of particles **completely from scratch, uniformly and randomly across the map**.
* This guarantees the robot a continuous ability to "explore" and recover even if it becomes completely lost.

---

## Practical Example: 2D Localization of the Duckiebot

Consider the scenario proposed in the original notebook:

* A robot moves in an obstacle-free room starting from the origin $(x, y) = (0, 0)$.
* The robot applies constant velocity commands $v_x = v_y = 0.1\text{ m/s}$ along the diagonal for $N = 100$ time steps.
* At each time step, the robot measures its position using wall-distance sensors.

### Motion Model

$$\mathbf{x}_t = \mathbf{x}_{t-1} + \mathbf{u}_t + \mathbf{w}_t, \quad \mathbf{w}_t \sim \mathcal{N}([0, 0]^T, \sigma_R I_2)$$


with motion noise standard deviation $\sigma_R = 0.01$.

### Measurement Model

$$\mathbf{z}_t = \mathbf{x}_t + \mathbf{n}_t, \quad \mathbf{n}_t \sim \mathcal{N}([0, 0]^T, \sigma_Q I_2)$$


with sensor noise standard deviation $\sigma_Q = 0.5$.

> **Fundamental Note:**
> Unlike the Kalman Filter, in the Particle Filter **there is no mathematical requirement that the models be linear or that the noise be Gaussian**! Simple models are used here to make the didactic comparison immediate, but a Particle Filter can handle any complex or trigonometric function.

### Code Explanation: Simulating Ground Truth and Noisy Measurements

* **Purpose:** Generates synthetic data representing the robot's true path and the noisy readings from its sensors.
* **Key Details:**
* `ground_truth_x` and `ground_truth_y` create a clean 45-degree diagonal path from $(0,0)$ to $(10,10)$ over `num_data = 100` steps.
* `np.random.normal` generates Gaussian sensor noise with a high standard deviation ($\sigma = 0.5$) and adds it to the true coordinates to create `measurement_x` and `measurement_y`.
* The `matplotlib` block plots the true path (black dashed line) against the raw sensor readings (red dots), visually demonstrating how scattered and unreliable the raw sensor data is.



As seen in the plot, the sensor measurements are widely scattered around the ideal diagonal. Next, we build the **Particle Filter** architecture to filter out this noise and estimate the robot's true position with high accuracy.

---

## Implementing the `Particle` Class and Resampling

We need to define two key elements:

1. The **`Particle`** class:
* Each particle stores its state coordinates `self.state = np.array([x, y])` and a weight `self.weight`.
* The **`predict(u_t)`** method: shifts the particle by $\mathbf{u}_t$ while adding Gaussian noise with standard deviation $\sigma_R = 0.01$.
* The **`update(z_t)`** method: calculates the likelihood of the real measurement $\mathbf{z}_t$ given the particle's position, using the multivariate Gaussian probability density function:
`scipy.stats.multivariate_normal.pdf(self.state, mean=z_t, cov=Q)`


2. The **`sample_particle(particles)`** function:
* Performs stochastic extraction (*Roulette Wheel Selection*): the probability of picking a particle is proportional to its normalized weight.



### Code Explanation: `Particle` Class & Roulette Wheel Sampling

* **Purpose:** Defines the core data structure for a single hypothesis (`Particle`) and the selection mechanism for resampling.
* **Key Details:**
* `sigma_R = 0.01` and `sigma_Q = 0.50` establish the trust balance: we trust our motion model (low noise) much more than our raw sensor readings (high noise).
* `Particle.predict(u_t)` updates `self.state` by adding the control vector `u_t` plus random Gaussian noise drawn from `sigma_R`.
* `Particle.update(z_t)` constructs the $2 \times 2$ covariance matrix $Q = \sigma_Q I_2$ and evaluates how close the particle's predicted state is to the actual sensor measurement `z_t` using a 2D Gaussian PDF.
* `sample_particle(particles)` implements **Roulette Wheel Selection**: it sums all weights, picks a random threshold `r` between `0` and `total_weight`, and iterates through the particles accumulating weights until it crosses `r`. Particles with larger weights occupy a larger "slice" of the wheel and are selected more frequently.

The `sample_particle(particles)` function selects and returns a single particle from a list using **weighted random sampling** (also known as *Roulette Wheel Selection*), where particles with higher weights have a higher probability of being chosen.

This version of `sample_particle(particles)` implements **Roulette Wheel Selection** using pure Python without relying on NumPy's built-in sampling functions.

* **Total Weight Calculation:**
```python
total = sum(particle.weight for particle in particles)

```


Computes the sum of all particle weights using a generator expression. This represents the total "circumference" of the roulette wheel.
* **Random Threshold Selection:**
```python
r = random.uniform(0, total)

```


Picks a random floating-point number `r` uniformly distributed between `0` and `total`. Think of `r` as the exact spot where the ball lands on the roulette wheel.
* **Cumulative Sum & Selection Loop:**
```python
upto = 0
for particle in particles:
    if upto + particle.weight >= r:
        return particle
    upto += particle.weight

```


Iterates through the particles while keeping a running sum (`upto`) of the weights seen so far. Each particle occupies a segment of length `particle.weight` along the interval `[0, total]`. As soon as the cumulative sum reaches or crosses the random threshold `r`, that particle is selected and returned. Particles with larger weights cover wider segments, making them proportionally more likely to capture `r`.
* **Floating-Point Fallback:**
```python
return particles[-1]

```


Acts as a safety net against floating-point precision errors. In rare cases, due to tiny rounding inaccuracies in floating-point addition, the final sum of `upto` might end up slightly less than `r` (e.g., `0.9999999999999999` vs. `1.0`). Returning the last particle (`particles[-1]`) guarantees the function always returns a valid particle instead of `None`.



---

## Running the Particle Filter Over Time

We now assemble the complete Particle Filter loop over the $100$ time steps:

* **$M = 100$ particles.**
* **$\alpha = 0.95$**: at each step, $95\%$ of the particles are resampled based on their weights, while $5\%$ are randomly regenerated to prevent *depletion*.
* At every time step:
1. Particles advance via `predict(u_t)`.
2. We acquire the sensor measurement $\mathbf{z}_t$.
3. We update weights via `update(z_t)`.
4. We build the new set of resampled particles (`sample_particle`).
5. We compute the final mean state estimate $(\hat{x}_t, \hat{y}_t)$.



### Code Explanation: The Main Simulation Loop

* **Purpose:** Executes the full localization loop across 100 time steps and logs the estimated trajectory.
* **Key Details:**
* **Initialization:** `initialize_particle()` spawns 100 particles uniformly across the $10 \times 10$ map with equal weights (`1.0`).
* **Step A (Predict):** Every particle moves forward by the constant control step `u_t = [0.1, 0.1]` plus motion noise.
* **Step B (Measure):** Simulates a new noisy sensor reading `z_t` around the robot's current ground-truth position.
* **Step C (Update):** Calls `p.update(z_t)` on every particle to score them against the new sensor reading.
* **Step D (Resampling with Anti-Depletion):** Draws `95` particles (`int(num_particles * alpha)`) using `sample_particle`, resetting their weights to `1.0` (using `copy.deepcopy` so duplicate particles don't share the same memory reference). The remaining `5` particles are spawned randomly across the map via `initialize_particle()`.
* **Step E (State Estimation):** Computes the robot's estimated $(x, y)$ position by taking the mean of the 95 resampled particles (explicitly excluding the 5 random exploratory particles so they don't skew the position estimate).



---

## Results: Comparing True Trajectory vs. Measurements vs. Particle Filter

We visualize the entire path estimated by the Particle Filter compared against the raw measurements and the ground truth:

### Code Explanation: Plotting the Results

* **Purpose:** Renders a 2D comparison of the ground truth (black dashed line), raw noisy measurements (red dots), and the Particle Filter's estimated trajectory (solid green line).
* **Key Details:** `plt.gca().set_aspect('equal', adjustable='box')` ensures that the X and Y axes share the same scale so the 45-degree diagonal path is not visually distorted.

---

## Why Is the Particle Filter Less Accurate at the Beginning? (Convergence Phenomenon)

If you look closely at the very first steps near the origin $(0, 0)$:

* The particles were randomly generated across the **entire** area from $0$ to $10$.
* At time $t=0$, the filter does not yet know where the robot is located!
* It takes a few time steps (and several consecutive sensor measurements) for distant particles to be eliminated and for the remaining ones to cluster densely around the Duckiebot.
* Once **convergence** occurs, the green line faithfully and smoothly tracks the true trajectory, effectively filtering out the massive scatter of the red dots!

We can verify this by computing the instantaneous localization error.

### Code Explanation: Calculating Instantaneous Localization Error

* **Purpose:** Quantifies the filter's performance by measuring the Euclidean distance error between `filtered_states` and `ground_truth` at each time step.
* **Key Details:** Typically computed using `np.linalg.norm(filtered_states - ground_truth[1:], axis=1)`, which shows a high initial error spike during the first few steps (global uncertainty) followed by a sharp drop and stabilization near zero once the particle cloud converges on the robot.